# Create Ontology Projection Tables

Creates managed Delta projection tables used by `ClinicalDeviceOntology`.

This notebook mirrors Phase 4 / Step 8b in `Deploy-All.ps1`: `DeviceAssociation`, `EncounterOntology`, `ConditionOntology`, `MedicationRequestOntology`, `ObservationOntology`, and `ImagingStudyOntology`.

`INCLUDE_FHIR` and `INCLUDE_DICOM` gate optional source families the same way `deploy-ontology.ps1` uses `-IncludeFhir` / `-IncludeDicom`. `DeviceAssociation` remains materialized for telemetry/device relationships.

Patient joins are normalized for HDS 1.3.x flattened references: most clinical tables prefer stripped `$.msftSourceReference`, then `$.identifier.value`, then `$.idOrig`; `ImagingStudy` prefers `$.identifier.value` first.

### Prerequisites
- Attach this notebook to the Silver Lakehouse containing the HDS tables.
- HDS clinical and imaging pipelines must have completed for the corresponding projection tables to populate.

### Re-run
Re-run after Silver clinical/imaging data refreshes and before refreshing the Fabric IQ ontology graph model.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import get_json_object
import sys

WORKSPACE_ID = ""  # Optional: set only if table-name resolution fails
LAKEHOUSE_ID = ""  # Optional: set only if table-name resolution fails
INCLUDE_FHIR = True   # Set False if FHIR source tables are intentionally skipped
INCLUDE_DICOM = True  # Set False if ImagingStudy source data is intentionally skipped

def read_delta_table(name):
    print(f"Attempting to load table {name} by name...")
    try:
        df = spark.read.table(name)
        print(f"Successfully loaded {name} table by name.")
        return df
    except Exception as e:
        print(f"Failed to load {name} table by name: {e}")
        print("Falling back to absolute OneLake ABFSS path...")
        if not WORKSPACE_ID or not LAKEHOUSE_ID:
            raise RuntimeError("Attach this notebook to the Silver Lakehouse or set WORKSPACE_ID and LAKEHOUSE_ID for ABFSS fallback.")
        path = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Tables/{name}"
        return spark.read.format("delta").load(path)

def save_delta_table(df, name):
    print(f"Attempting to save table {name} by name...")
    try:
        df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(name)
        print(f"Successfully saved {name} table by name.")
    except Exception as e:
        print(f"Failed to save {name} table by name: {e}")
        print("Falling back to absolute OneLake ABFSS path...")
        if not WORKSPACE_ID or not LAKEHOUSE_ID:
            raise RuntimeError("Attach this notebook to the Silver Lakehouse or set WORKSPACE_ID and LAKEHOUSE_ID for ABFSS fallback.")
        path = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Tables/{name}"
        df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(path)
        print(f"{name} table materialized successfully via abfss fallback.")

def present(df, name):
    return name in df.columns

def maybe_col(df, name, data_type="string"):
    return F.col(name) if present(df, name) else F.lit(None).cast(data_type)

def select_col(df, name, data_type="string"):
    return maybe_col(df, name, data_type).alias(name)

def json_col(df, name, path):
    return get_json_object(F.col(name), path) if present(df, name) else F.lit(None).cast("string")

def non_empty(expr):
    return F.when(expr.isNotNull() & (F.length(F.trim(expr.cast("string"))) > 0), expr.cast("string"))

def patient_id_expr(df, column_name, imaging=False):
    # HDS 1.3.x clears FHIR Reference.reference and preserves the original
    # resource link under msftSourceReference, with idOrig as the normalized UUID.
    # ImagingStudy also receives identifier.value and should prefer it first.
    identifier = non_empty(json_col(df, column_name, "$.identifier.value"))
    msft = non_empty(F.regexp_replace(json_col(df, column_name, "$.msftSourceReference"), "^Patient/", ""))
    id_orig = non_empty(json_col(df, column_name, "$.idOrig"))
    if imaging:
        return F.coalesce(identifier, msft, id_orig)
    return F.coalesce(msft, identifier, id_orig)


basic_df = read_delta_table("Basic")
device_df = basic_df.filter(get_json_object(F.col("code_string"), "$.coding[0].code") == "device-assoc")
device_association_df = device_df.select(
    select_col(device_df, "id"),
    select_col(device_df, "idOrig"),
    F.regexp_replace(json_col(device_df, "extension", "$[0].valueReference.reference"), "^Device/", "").alias("device_ref"),
    json_col(device_df, "subject_string", "$.display").alias("patient_name"),
    patient_id_expr(device_df, "subject_string").alias("patient_id"),
    json_col(device_df, "code_string", "$.coding[0].code").alias("assoc_code"),
    json_col(device_df, "code_string", "$.coding[0].display").alias("assoc_display")
)
save_delta_table(device_association_df, "DeviceAssociation")

if INCLUDE_FHIR:
    encounter_df = read_delta_table("Encounter")
    save_delta_table(encounter_df.select(
        select_col(encounter_df, "idOrig"),
        select_col(encounter_df, "class_string"),
        select_col(encounter_df, "status"),
        F.coalesce(non_empty(maybe_col(encounter_df, "period_start")), non_empty(json_col(encounter_df, "period_string", "$.start"))).alias("period_start"),
        patient_id_expr(encounter_df, "subject_string").alias("patient_id")
    ), "EncounterOntology")

    condition_df = read_delta_table("Condition")
    save_delta_table(condition_df.select(
        select_col(condition_df, "idOrig"),
        select_col(condition_df, "code_string"),
        select_col(condition_df, "clinicalStatus_string"),
        patient_id_expr(condition_df, "subject_string").alias("patient_id")
    ), "ConditionOntology")

    med_df = read_delta_table("MedicationRequest")
    save_delta_table(med_df.select(
        select_col(med_df, "idOrig"),
        select_col(med_df, "medicationCodeableConcept_string"),
        select_col(med_df, "status"),
        select_col(med_df, "authoredOn"),
        patient_id_expr(med_df, "subject_string").alias("patient_id")
    ), "MedicationRequestOntology")

    obs_df = read_delta_table("Observation")
    save_delta_table(obs_df.select(
        select_col(obs_df, "idOrig"),
        select_col(obs_df, "code_string"),
        select_col(obs_df, "valueQuantity_value", "double"),
        select_col(obs_df, "valueQuantity_unit"),
        select_col(obs_df, "effectiveDateTime"),
        patient_id_expr(obs_df, "subject_string").alias("patient_id")
    ), "ObservationOntology")
else:
    print("Skipping FHIR ontology projection tables because INCLUDE_FHIR is false.")

if INCLUDE_DICOM:
    img_df = read_delta_table("ImagingStudy")
    save_delta_table(img_df.select(
        select_col(img_df, "idOrig"),
        select_col(img_df, "description"),
        patient_id_expr(img_df, "subject_string", imaging=True).alias("patient_id"),
        select_col(img_df, "numberOfSeries", "long"),
        select_col(img_df, "numberOfInstances", "long")
    ), "ImagingStudyOntology")
else:
    print("Skipping ImagingStudyOntology because INCLUDE_DICOM is false.")

print("Ontology projection tables materialized successfully.")


### Verify
Counts are checked only for projection tables enabled by `INCLUDE_FHIR` / `INCLUDE_DICOM`.


In [ ]:
verify_tables = ["DeviceAssociation"]
if INCLUDE_FHIR:
    verify_tables.extend(["EncounterOntology", "ConditionOntology", "MedicationRequestOntology", "ObservationOntology"])
if INCLUDE_DICOM:
    verify_tables.append("ImagingStudyOntology")

counts = []
for table_name in verify_tables:
    row_count = spark.read.table(table_name).count()
    counts.append((table_name, row_count))

spark.createDataFrame(counts, ["table_name", "row_count"]).show(truncate=False)
